# DA5301W: Python for Data Science
## Assignment 2 (Take-Home): Social Network Graph Search

**Department of Data Science and AI (DSAI), IIT Madras**  
**Topic:** Graph Search on SNAP Facebook Friendship Network  
**Total Marks:** 100 Marks  

---
### Overview & Guidelines
- **Dataset**: Stanford SNAP Facebook Friendship Network (`facebook_combined.txt`).
- **Exercise 1 [40 Marks]**: Uninformed Search — Breadth-First Search (BFS) and Depth-First Search (DFS).
  - *Note: You may directly reuse or adapt your BFS and DFS implementations from the In-Class assignment.*
- **Exercise 2 [60 Marks]**: Informed Search — A* Search on Weighted Social Network with Jaccard Friction Weights.
- **Evaluation**: Evaluated strictly on the 2 benchmark test cases per algorithm (No private test cases, no external plotting required).

## Exercise 0: Setup and Dataset Ingestion with NumPy

Load `data/facebook_combined.txt` using `np.loadtxt` and verify network statistics.

**Dataset note:** The setup cell downloads and extracts the SNAP file if it is not already present. Keep the notebook in `TakeHome/src/` and the dataset in `TakeHome/data/`; the code supports running from either the project root or `src/`.


In [7]:
%pip install networkx matplotlib numpy

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import gzip
import urllib.request
import heapq
from collections import deque

import numpy as np
import networkx as nx

# Dataset location. The assignment expects data/facebook_combined.txt.
# The notebook may be launched from the project root or from the src/ folder.
if os.path.isdir("data"):
    data_dir = "data"
elif os.path.isdir(os.path.join("..", "data")):
    data_dir = os.path.join("..", "data")
else:
    data_dir = "data"
os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, "facebook_combined.txt")
compressed_path = os.path.join(data_dir, "facebook_combined.txt.gz")

# Download and extract the SNAP dataset if the text file is not present.
if not os.path.exists(data_path):
    if not os.path.exists(compressed_path):
        dataset_url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
        print("Downloading the SNAP Facebook dataset ...")
        urllib.request.urlretrieve(dataset_url, compressed_path)
    print("Extracting the dataset ...")
    with gzip.open(compressed_path, "rb") as source, open(data_path, "wb") as target:
        target.write(source.read())

# Load the edge list with NumPy. Each row is an undirected edge (u, v).
edges = np.loadtxt(data_path, dtype=np.int64)
edges = np.atleast_2d(edges)

print(f"Loaded {len(edges):,} edges using np.loadtxt.")
print(f"Node ID range: min = {edges.min()}, max = {edges.max()}")

# Build an undirected adjacency dictionary and sort every neighbor list.
node_ids = np.unique(edges)
adj = {int(node): set() for node in node_ids}
for u, v in edges:
    adj[int(u)].add(int(v))
    adj[int(v)].add(int(u))
adj = {node: sorted(neighbors) for node, neighbors in adj.items()}

# Verify the graph statistics with NetworkX.
G_social = nx.Graph()
G_social.add_edges_from((int(u), int(v)) for u, v in edges)
print("NetworkX verification")
print("Number of nodes:", G_social.number_of_nodes())
print("Number of undirected edges:", G_social.number_of_edges())

assert G_social.number_of_nodes() == 4039, "Unexpected number of nodes."
assert G_social.number_of_edges() == 88234, "Unexpected number of undirected edges."
assert min(adj) == 0 and max(adj) == 4038, "Unexpected node ID range."


Loaded 88,234 edges using np.loadtxt.
Node ID range: min = 0, max = 4038
NetworkX verification
Number of nodes: 4039
Number of undirected edges: 88234


## Exercise 1: Uninformed Search — BFS and DFS [40 Marks]

*Note: You may directly reuse and adapt your `bfs` and `dfs` implementations from the In-Class assignment.*

1. **BFS Evaluation** [20 Marks]: `bfs(graph, start, goal)` returning `(path, expansions)` [10 Marks per test case].
2. **DFS Evaluation** [20 Marks]: `dfs(graph, start, goal)` returning `(path, expansions)` [10 Marks per test case].

**Test Cases**:
- **Test Case 1**: `(107, 1684)` (Direct shortcut between top two hubs)
- **Test Case 2**: `(107, 692)` (Referral route to the furthest node in the network)

*Note: Inspect neighbors in ascending order of node ID for deterministic expansion order.*

In [9]:
# BFS and DFS return (path, expansions).
# An expansion is counted when a node is removed from the queue/stack for processing.

def reconstruct_path(parent, goal):
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    return path[::-1]


def bfs(graph, start, goal):
    """Breadth-First Search using a FIFO queue."""
    queue = deque([start])
    parent = {start: None}
    discovered = {start}
    expansions = 0

    while queue:
        u = queue.popleft()
        expansions += 1

        if u == goal:
            return reconstruct_path(parent, goal), expansions

        for v in sorted(graph[u]):
            if v not in discovered:
                discovered.add(v)
                parent[v] = u
                queue.append(v)

    return [], expansions


def dfs(graph, start, goal):
    """Iterative Depth-First Search using an explicit stack."""
    stack = [start]
    parent = {start: None}
    discovered = {start}
    expansions = 0

    while stack:
        u = stack.pop()
        expansions += 1

        if u == goal:
            return reconstruct_path(parent, goal), expansions

        # Reverse push order so the smallest neighbor is popped first.
        for v in sorted(graph[u], reverse=True):
            if v not in discovered:
                discovered.add(v)
                parent[v] = u
                stack.append(v)

    return [], expansions


# Run both algorithms on the two benchmark pairs.
test_pairs = [("pair_1", 107, 1684), ("pair_2", 107, 692)]
bfs_results = {}
dfs_results = {}

for label, start, goal in test_pairs:
    path, expansions = bfs(adj, start, goal)
    bfs_results[label] = (path, len(path) - 1, expansions)
    print(f"BFS {label} ({start} -> {goal})")
    print("Path:", path)
    print("Path length (hops):", len(path) - 1)
    print("Expansion count:", expansions)

for label, start, goal in test_pairs:
    path, expansions = dfs(adj, start, goal)
    dfs_results[label] = (path, len(path) - 1, expansions)
    print(f"\nDFS {label} ({start} -> {goal})")
    print("Path:", path)
    print("Path length (hops):", len(path) - 1)
    print("Expansion count:", expansions)


BFS pair_1 (107 -> 1684)
Path: [107, 1684]
Path length (hops): 1
Expansion count: 819
BFS pair_2 (107 -> 692)
Path: [107, 1085, 862, 698, 686, 692]
Path length (hops): 5
Expansion count: 3903

DFS pair_1 (107 -> 1684)
Path: [107, 1684]
Path length (hops): 1
Expansion count: 3796

DFS pair_2 (107 -> 692)
Path: [107, 0, 173, 427, 425, 373, 567, 3437, 698, 686, 692]
Path length (hops): 10
Expansion count: 1188


## Exercise 2: Informed Search — A* on the Weighted Social Network [60 Marks]

To prepare the social graph for A* search, compute edge weights using neighborhood Jaccard similarity.

### Pre-Search Step: Jaccard Similarity & Friction Weights [20 Marks]
1. Compute the neighborhood Jaccard similarity for every edge:
   $$J(u, v) = \frac{|N(u) \cap N(v)|}{|N(u) \cup N(v)|}$$
2. Assign integer friction weights:
   $$w(u, v) = 1 + \lfloor 10 \times (1 - J(u, v)) \rfloor \in [1, 11]$$
3. Print summary statistics of friction weights (min, max, mean).

> **Concrete Example**:
> If $N(u) = \{1, 2, 3, 4\}$ and $N(v) = \{3, 4, 5, 6, 7\}$:
> - Intersection $|N(u) \cap N(v)| = |\{3, 4\}| = 2$
> - Union $|N(u) \cup N(v)| = |\{1, 2, 3, 4, 5, 6, 7\}| = 7$
> - Jaccard $J(u, v) = 2/7 \approx 0.2857$
> - Friction weight: $w(u, v) = 1 + \lfloor 10 \times (1 - 0.2857) \rfloor = 1 + 7 = 8$.

### A* Search Implementation & 2 Benchmark Tests [40 Marks]
1. Define modular helper subroutines `extract_min(open_set)` and `reconstruct_path(parent, goal)` [10 Marks].
2. Define the **Admissible Neighborhood Heuristic** $h(u, t)$:
   $$h(u, t) = \begin{cases} 0 & \text{if } u = t \\ w(u, t) & \text{if } (u, t) \in E \\ 2 & \text{if } (u, t) \notin E \end{cases}$$
3. Implement `astar_search(graph, weights, start, goal, heuristic_fn)`.
4. Evaluate on Test Case 1 `(107, 1684)` [15 Marks] and Test Case 2 `(107, 692)` [15 Marks].

In [10]:
# Degree of each node using NumPy array operations.
degree = np.bincount(edges.ravel(), minlength=4039)
top_hub_ids = np.argsort(-degree, kind="stable")[:3]
print("Top 3 highest-degree hubs (node ID, degree):")
for node in top_hub_ids:
    print(int(node), int(degree[node]))
print("Mean degree:", float(degree.mean()))

# Compute Jaccard similarity and integer friction weight for every
# undirected edge. Store both directions because the graph is undirected.
weights = {}
jaccard_values = []
friction_values = []

for u, v in edges:
    u, v = int(u), int(v)
    neighbors_u = set(adj[u])
    neighbors_v = set(adj[v])

    intersection_size = len(neighbors_u & neighbors_v)
    union_size = len(neighbors_u | neighbors_v)
    similarity = intersection_size / union_size if union_size else 0.0

    friction = 1 + int(np.floor(10 * (1 - similarity)))
    weights[(u, v)] = friction
    weights[(v, u)] = friction

    jaccard_values.append(similarity)
    friction_values.append(friction)

friction_values = np.asarray(friction_values, dtype=np.int64)
print("\nSocial friction weight statistics (one value per undirected edge)")
print("Minimum weight:", int(friction_values.min()))
print("Maximum weight:", int(friction_values.max()))
print("Mean weight:", float(friction_values.mean()))


Top 3 highest-degree hubs (node ID, degree):
107 1045
1684 792
1912 755
Mean degree: 43.69101262688784

Social friction weight statistics (one value per undirected edge)
Minimum weight: 1
Maximum weight: 11
Mean weight: 6.818165333091552


In [11]:
def extract_min(open_set):
    """Remove and return the minimum (f, g, node) tuple from the min-heap."""
    return heapq.heappop(open_set)


def reconstruct_path(parent, goal):
    """Reconstruct the source-to-goal path using parent pointers."""
    path = [goal]
    current = goal
    while parent[current] is not None:
        current = parent[current]
        path.append(current)
    return path[::-1]


def heuristic(u, goal, weights=None):
    """Admissible-neighborhood heuristic specified in the assignment."""
    if u == goal:
        return 0
    if (u, goal) in weights:
        return weights[(u, goal)]
    return 2


def astar_search(graph, weights, start, goal, heuristic_fn=None):
    """Return (path, total_cost, expansions) using A* graph search."""
    if heuristic_fn is None:
        heuristic_fn = heuristic

    g_score = {node: float("inf") for node in graph}
    g_score[start] = 0
    parent = {start: None}

    # Heap entries are (f = g + h, g, node_id); node ID breaks ties.
    open_set = [(heuristic_fn(start, goal, weights), 0, start)]
    closed = set()
    expansions = 0

    while open_set:
        f_value, current_g, u = extract_min(open_set)

        if u in closed:
            continue
        # Ignore an outdated heap entry if a better g-score was found later.
        if current_g != g_score[u]:
            continue

        closed.add(u)
        expansions += 1

        if u == goal:
            return reconstruct_path(parent, goal), g_score[goal], expansions

        for v in sorted(graph[u]):
            if v in closed:
                continue

            new_g = g_score[u] + weights[(u, v)]
            if new_g < g_score[v]:
                g_score[v] = new_g
                parent[v] = u
                new_f = new_g + heuristic_fn(v, goal, weights)
                heapq.heappush(open_set, (new_f, new_g, v))

    return [], float("inf"), expansions


# Benchmark A* on the two assigned test cases.
astar_results = {}
for label, start, goal in test_pairs:
    path, total_cost, expansions = astar_search(
        adj, weights, start, goal, heuristic
    )
    astar_results[label] = (path, total_cost, expansions)
    print(f"A* {label} ({start} -> {goal})")
    print("Optimal path:", path)
    print("Total path cost:", total_cost)
    print("Path length (hops):", len(path) - 1)
    print("Expansion count:", expansions)
    print()


A* pair_1 (107 -> 1684)
Optimal path: [107, 1684]
Total path cost: 10
Path length (hops): 1
Expansion count: 10

A* pair_2 (107 -> 692)
Optimal path: [107, 1085, 862, 698, 686, 692]
Total path cost: 49
Path length (hops): 5
Expansion count: 3986



## Summary of Benchmark Results

Performance comparison across all algorithms on the 2 assigned test cases.

In [12]:
# Benchmark summary: BFS/DFS optimize hop count; A* optimizes friction cost.
print("=" * 86)
print("BENCHMARK SUMMARY (2 TESTS PER ALGORITHM)")
print("=" * 86)
print(f"{'Algorithm':<12} {'Test Case':<12} {'Length (Hops)':>15} {'Cost':>12} {'Expansions':>13}")
print("-" * 86)

for label, _, _ in test_pairs:
    path, hops, expansions = bfs_results[label]
    print(f"{'BFS':<12} {label:<12} {hops:>15} {'-':>12} {expansions:>13}")

for label, _, _ in test_pairs:
    path, hops, expansions = dfs_results[label]
    print(f"{'DFS':<12} {label:<12} {hops:>15} {'-':>12} {expansions:>13}")

for label, _, _ in test_pairs:
    path, cost, expansions = astar_results[label]
    hops = len(path) - 1
    print(f"{'A*':<12} {label:<12} {hops:>15} {cost:>12} {expansions:>13}")

print("=" * 86)


BENCHMARK SUMMARY (2 TESTS PER ALGORITHM)
Algorithm    Test Case      Length (Hops)         Cost    Expansions
--------------------------------------------------------------------------------------
BFS          pair_1                     1            -           819
BFS          pair_2                     5            -          3903
DFS          pair_1                     1            -          3796
DFS          pair_2                    10            -          1188
A*           pair_1                     1           10            10
A*           pair_2                     5           49          3986
